## Open notebook in:
| Colab                                 |  
|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH04/ch04_quantize_T2I_models.ipynb)                                                        

# About this notebook

This notebook demonstrates a memory-efficient approach to generating high-quality images using the PixArt-Σ model, a state-of-the-art diffusion transformer for ultra-high-resolution image synthesis. As deep learning models grow in complexity, memory management becomes a crucial aspect, especially when working with limited GPU resources.

In this example, you use advanced quantization techniques provided by the [`BitsAndBytesConfig`](https://huggingface.co/docs/transformers/main/en/quantization/bitsandbytes) configuration and the [`optimum.quanto`](https://github.com/huggingface/optimum-quanto) library to reduce the memory footprint while maintaining performance. The notebook will guide you through the steps of setting up a quantized text encoder, generating prompt embeddings, and efficiently managing GPU memory during the image generation process.

You will also monitor GPU memory usage throughout the process, using [PyTorch's memory functionality](https://pytorch.org/docs/stable/torch_cuda_memory.html#), and employ strategies like freezing parts of the model and cleaning up unused resources to further optimize memory consumption. By the end of this notebook, you will have a practical understanding of how to handle large-scale models on limited hardware, enabling you to generate high-quality images with reduced memory overhead.
The provided code is inspired by the [examples](https://github.com/huggingface/optimum-quanto/blob/main/examples/vision/text-to-image/quantize_pixart_sigma.py) in Hugging Face's quanto libary and [Diffusers library](https://huggingface.co/docs/diffusers/main/en/api/pipelines/pixart_sigma).


#Installs

In [ ]:
!pip install -qU transformers==5.9.0 diffusers==0.38.0 bitsandbytes==0.49.2 optimum[quanto]==2.1.0 accelerate==1.13.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.1/509.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 16.8 MB/s eta 0:00:00


In [ ]:
!pip freeze | grep -E 'transformers|diffusers|bitsandbytes|optimum|accelerate'

accelerate==1.13.0
bitsandbytes==0.49.2
diffusers==0.38.0
optimum==2.1.0
optimum-quanto==0.2.7
sentence-transformers==5.5.1
transformers==5.9.0


In [ ]:
#!pip install accelerate==0.33.0 -qqq

#Imports

In [ ]:
from transformers import T5EncoderModel, BitsAndBytesConfig
from diffusers import PixArtSigmaPipeline
from optimum.quanto import freeze, qfloat8, qint4, qint8, quantize
import torch
import gc

Multiple distributions found for package optimum. Picked distribution: optimum
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


# Helper Function

In [ ]:
def to_giga_bytes(bytes):
    return bytes / (1024 ** 3)


# Quantize the text encoder model

In [ ]:
torch.cuda.memory._record_memory_history()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

text_encoder = T5EncoderModel.from_pretrained(
    "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS",
    subfolder="text_encoder",
    quantization_config=quant_config,
    device_map="balanced",
)

pipe = PixArtSigmaPipeline.from_pretrained(
    "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS",
    text_encoder=text_encoder,
    transformer=None,
    device_map="balanced"
)

with torch.no_grad():
    prompt = "Cute animated tabby with big eyes"
    prompt_embeds, prompt_attention_mask, negative_embeds, negative_prompt_attention_mask = pipe.encode_prompt(prompt)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


model_index.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
print(
    f"Max memory allocated: {to_giga_bytes(torch.cuda.max_memory_allocated())} GB"
)

print(
    f"Max memory reserved: {to_giga_bytes(torch.cuda.memory_reserved())} GB"
)

Max memory allocated: 6.596279144287109 GB
Max memory reserved: 6.685546875 GB


# Delete Text Encoder

In [ ]:
del text_encoder
del pipe

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print(
    f"Max memory allocated: {to_giga_bytes(torch.cuda.max_memory_allocated())} GB"
)

print(
    f"Max memory reserved: {to_giga_bytes(torch.cuda.memory_reserved())} GB"
)

Max memory allocated: 6.596279144287109 GB
Max memory reserved: 5.931640625 GB


# Quantize the Diffusion Model

In [ ]:
pipe = PixArtSigmaPipeline.from_pretrained(
    "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS",
    text_encoder=None,
    torch_dtype=torch.float16,
).to("cuda")

quantize(pipe.transformer, weights=qint8, exclude="proj_out")
freeze(pipe.transformer)

latents = pipe(
    negative_prompt=None,
    prompt_embeds=prompt_embeds,
    negative_prompt_embeds=negative_embeds,
    prompt_attention_mask=prompt_attention_mask,
    negative_prompt_attention_mask=negative_prompt_attention_mask,
    num_images_per_prompt=1,
    output_type="latent",
).images

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
print(
    f"Max memory allocated: {to_giga_bytes(torch.cuda.max_memory_allocated())} GB"
)

print(
    f"Max memory reserved: {to_giga_bytes(torch.cuda.memory_reserved())} GB"
)

Max memory allocated: 6.596279144287109 GB
Max memory reserved: 5.943359375 GB


# Flush the Memory

In [ ]:
del pipe.transformer


In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print(
    f"Max memory allocated: {to_giga_bytes(torch.cuda.max_memory_allocated())} GB"
)

print(
    f"Max memory reserved: {to_giga_bytes(torch.cuda.memory_reserved())} GB"
)

Max memory allocated: 6.596279144287109 GB
Max memory reserved: 5.943359375 GB


# Generate the Image

In [ ]:
with torch.no_grad():
    image = pipe.vae.decode(latents / pipe.vae.config.scaling_factor, return_dict=False)[0]
image = pipe.image_processor.postprocess(image, output_type="pil")

image[0].save("tabby.png")

# Get Memory Summary

In [ ]:
torch.cuda.memory._dump_snapshot("PixArtSigma_quant.pickle")

print(
    torch.cuda.memory_summary()
)


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      | 183661 KiB |   6754 MiB | 557101 MiB | 556922 MiB |
|       from large pool | 173216 KiB |   6682 MiB | 556402 MiB | 556233 MiB |
|       from small pool |  10445 KiB |     75 MiB |    699 MiB |    688 MiB |
|---------------------------------------------------------------------------|
| Active memory         | 183661 KiB |   6754 MiB | 557101 MiB | 556922 MiB |
|       from large pool | 173216 KiB |   6682 MiB | 556402 MiB |